# Ders 4B — SHAP, SHAP Waterfall ve LIME ile Model Açıklama

Bu notebookta modelin neden karar verdiğini SHAP beeswarm, SHAP waterfall ve LIME ile açıklayacağız.

## 1. Paket kurulumu

Bu hücre Colab ortamında eksik paketleri kurar. RDKit moleküler fingerprint üretmek için gereklidir.

In [ ]:
# Google Colab için gerekli paketleri kuruyoruz.
# RDKit moleküllerden fingerprint üretmek için gerekir.
# SHAP/LIME yalnızca ilgili notebookta kullanılacak; burada temel paketleri kontrol ediyoruz.

import sys, subprocess, importlib.util

def pip_install(package_name):
    print(f"[INSTALL] {package_name} kuruluyor...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", package_name])

for pkg, pip_name in [
    ("pandas", "pandas"),
    ("numpy", "numpy"),
    ("sklearn", "scikit-learn"),
    ("matplotlib", "matplotlib"),
    ("joblib", "joblib"),
]:
    if importlib.util.find_spec(pkg) is None:
        pip_install(pip_name)

# RDKit Colab ortamında bazen rdkit, bazen rdkit-pypi ile kuruluyor.
if importlib.util.find_spec("rdkit") is None:
    try:
        pip_install("rdkit")
    except Exception:
        pip_install("rdkit-pypi")

print("✅ Paket kontrolü tamamlandı.")

## 2. Paketleri çağırma

Bu hücrede analiz boyunca kullanacağımız paketleri import ediyoruz.

In [ ]:
# Bu hücrede kullanacağımız Python paketlerini çağırıyoruz.
# pandas: tablo/veri işlemleri
# numpy: sayısal işlemler
# matplotlib: grafik çizimi
# scikit-learn: makine öğrenmesi modelleri ve metrikler
# RDKit: SMILES -> moleküler fingerprint dönüşümü

from pathlib import Path
import os
import json
import math
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import joblib

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate
from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score, average_precision_score,
    f1_score, precision_score, recall_score, roc_auc_score,
    confusion_matrix, ConfusionMatrixDisplay,
    RocCurveDisplay, PrecisionRecallDisplay
)
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import VarianceThreshold

from rdkit import Chem, DataStructs
from rdkit.Chem import MACCSkeys, rdFingerprintGenerator, Descriptors
from rdkit.ML.Descriptors.MoleculeDescriptors import MolecularDescriptorCalculator

print("✅ Importlar tamamlandı.")

## 3. Genel ayarlar

GitHub veri linkleri, target/SMILES kolonları ve çıktı klasörü burada tanımlanır.

In [ ]:
# ============================================================
# CONFIG
# ============================================================
# Bu iki veri seti GitHub üzerinden okunacak.
# GitHub'daki /blob/main/ linkleri raw linke otomatik çevriliyor.

DATA_URLS = {
    "A14_A17_ERa_BLA": "https://github.com/MOL-FEST/MOL_FEST_2026/blob/main/Train_2_1_A14_A17_ERa_BLA_agonist_antagonist.csv",
    "A15_A18_ERa_LUC_VM7": "https://github.com/MOL-FEST/MOL_FEST_2026/blob/main/Train_2_2_A15_A18_ERa_LUC_VM7_agonist_antagonist.csv",
}

# Ana analiz için default veri seti.
# İstersen bunu "A14_A17_ERa_BLA" yapabilirsin.
ACTIVE_DATASET_KEY = "A15_A18_ERa_LUC_VM7"

TARGET_COLUMN = "binary_label_agonist1_antagonist0"
SMILES_COLUMN = "QSAR-Ready SMILES"

RANDOM_STATE = 42
TEST_SIZE = 0.20

# Morgan fingerprint ayarları.
MORGAN_BITS = 1024
MORGAN_RADIUS = 2

# Çıktılar bu klasöre kaydedilecek.
OUTPUT_ROOT = Path("molfest_outputs")
OUTPUT_ROOT.mkdir(exist_ok=True)

print("✅ Config hazır.")
print(f"Aktif veri seti: {ACTIVE_DATASET_KEY}")

## 4. Ortak yardımcı fonksiyonlar

Bu fonksiyonlar veri okuma, fingerprint üretme, model eğitme, metrik hesaplama ve çıktı kaydetme işlerini kolaylaştırır.

In [ ]:
# ============================================================
# ORTAK YARDIMCI FONKSİYONLAR
# ============================================================

def note(title, message=""):
    """Notebook çalışırken açıklayıcı bilgi yazdırmak için küçük yardımcı."""
    print("\n" + "=" * 90)
    print(title)
    print("=" * 90)
    if message:
        print(message)


def github_to_raw(url):
    """GitHub blob linkini raw CSV linkine çevirir."""
    if "github.com" in url and "/blob/" in url:
        url = url.replace("https://github.com/", "https://raw.githubusercontent.com/")
        url = url.replace("/blob/", "/")
    return url


def read_csv_flexible(url_or_path):
    """CSV ayıracını otomatik anlamaya çalışır: önce ';', sonra ','."""
    source = github_to_raw(str(url_or_path))
    last_error = None
    for sep in [";", ","]:
        try:
            df = pd.read_csv(source, sep=sep, encoding="utf-8-sig", low_memory=False)
            # Yanlış ayraçta genellikle tek kolon gelir.
            if df.shape[1] > 1:
                return df, sep, source
        except Exception as e:
            last_error = e
    raise RuntimeError(f"CSV okunamadı: {url_or_path}\nSon hata: {last_error}")


def detect_column(df, preferred, keywords):
    if preferred in df.columns:
        return preferred
    for c in df.columns:
        cl = c.lower()
        if any(k.lower() in cl for k in keywords):
            return c
    raise ValueError(f"Kolon bulunamadı. Beklenen: {preferred}. Mevcut kolonlar: {list(df.columns)}")


def load_dataset(dataset_key=ACTIVE_DATASET_KEY):
    """GitHub'dan veri setini okur, target/smiles kolonlarını kontrol eder."""
    if dataset_key not in DATA_URLS:
        raise ValueError(f"dataset_key şunlardan biri olmalı: {list(DATA_URLS.keys())}")

    df, sep, source = read_csv_flexible(DATA_URLS[dataset_key])
    target_col = detect_column(df, TARGET_COLUMN, ["binary_label", "label", "target", "class"])
    smiles_col = detect_column(df, SMILES_COLUMN, ["smiles"])

    note(
        "Veri okundu",
        f"Kaynak: {source}\n"
        f"Ayraç: {repr(sep)}\n"
        f"Satır sayısı: {df.shape[0]}\n"
        f"Kolon sayısı: {df.shape[1]}\n"
        f"Target kolonu: {target_col}\n"
        f"SMILES kolonu: {smiles_col}"
    )

    return df, target_col, smiles_col, sep, source


def prepare_target_and_smiles(df, target_col, smiles_col):
    """Target ve SMILES temizliği: eksik target ve eksik SMILES çıkarılır."""
    before = len(df)
    y_num = pd.to_numeric(df[target_col], errors="coerce")
    mask = y_num.notna() & df[smiles_col].notna()

    df2 = df.loc[mask].copy().reset_index(drop=True)
    y = pd.to_numeric(df2[target_col], errors="coerce").astype(int).to_numpy()

    classes = np.unique(y)
    if len(classes) != 2:
        raise ValueError(f"Binary classification bekleniyor; bulunan sınıflar: {classes}")

    # Eğer sınıflar 0/1 değilse küçük sınıfı 0, büyük sınıfı 1 yapıyoruz.
    if not set(classes).issubset({0, 1}):
        mapping = {classes[0]: 0, classes[1]: 1}
        y = np.array([mapping[v] for v in y], dtype=int)

    note(
        "Target ve SMILES temizliği",
        f"Önceki satır sayısı: {before}\n"
        f"Kullanılabilir satır sayısı: {len(df2)}\n"
        f"Çıkarılan satır sayısı: {before - len(df2)}\n"
        f"Sınıf dağılımı: {dict(pd.Series(y).value_counts().sort_index())}"
    )
    return df2, y


def plot_class_distribution(y, out_path=None, title="Sınıf dağılımı"):
    counts = pd.Series(y).value_counts().sort_index()
    plt.figure(figsize=(6, 4))
    plt.bar([str(i) for i in counts.index], counts.values)
    plt.xlabel("Sınıf")
    plt.ylabel("Molekül sayısı")
    plt.title(title)
    plt.tight_layout()
    if out_path is not None:
        plt.savefig(out_path, dpi=300, bbox_inches="tight")
        print(f"[Kaydedildi] {out_path}")
    plt.show()


def smiles_to_morgan(smiles, radius=MORGAN_RADIUS, n_bits=MORGAN_BITS):
    generator = rdFingerprintGenerator.GetMorganGenerator(radius=radius, fpSize=n_bits)
    names = [f"Morgan_r{radius}_{i}" for i in range(n_bits)]
    rows, valid = [], []

    for smi in smiles:
        mol = Chem.MolFromSmiles(str(smi)) if pd.notna(smi) else None
        if mol is None:
            valid.append(False)
            continue
        fp = generator.GetFingerprint(mol)
        arr = np.zeros((n_bits,), dtype=np.float32)
        DataStructs.ConvertToNumpyArray(fp, arr)
        rows.append(arr)
        valid.append(True)

    return np.vstack(rows).astype(np.float32), names, np.array(valid, dtype=bool)


def smiles_to_maccs(smiles):
    names = [f"MACCS_{i}" for i in range(1, 167)]
    rows, valid = [], []

    for smi in smiles:
        mol = Chem.MolFromSmiles(str(smi)) if pd.notna(smi) else None
        if mol is None:
            valid.append(False)
            continue
        fp = MACCSkeys.GenMACCSKeys(mol)
        arr167 = np.zeros((167,), dtype=np.float32)
        DataStructs.ConvertToNumpyArray(fp, arr167)
        rows.append(arr167[1:])  # bit 0 gerçek MACCS key değildir
        valid.append(True)

    return np.vstack(rows).astype(np.float32), names, np.array(valid, dtype=bool)


def smiles_to_rdkit_descriptors(smiles):
    descriptor_names = [name for name, _ in Descriptors._descList]
    calc = MolecularDescriptorCalculator(descriptor_names)
    rows, valid = [], []

    for smi in smiles:
        mol = Chem.MolFromSmiles(str(smi)) if pd.notna(smi) else None
        if mol is None:
            valid.append(False)
            continue
        desc = np.array(calc.CalcDescriptors(mol), dtype=np.float32)
        desc = np.nan_to_num(desc, nan=0.0, posinf=0.0, neginf=0.0)
        rows.append(desc)
        valid.append(True)

    return np.vstack(rows).astype(np.float32), descriptor_names, np.array(valid, dtype=bool)


def build_feature_matrix(df, y, smiles_col, feature_set="morgan"):
    """SMILES'tan istenen feature setini üretir."""
    smiles = df[smiles_col].tolist()

    if feature_set == "morgan":
        X, names, valid = smiles_to_morgan(smiles)
    elif feature_set == "maccs":
        X, names, valid = smiles_to_maccs(smiles)
    elif feature_set == "rdkit_descriptors":
        X, names, valid = smiles_to_rdkit_descriptors(smiles)
    elif feature_set == "maccs_morgan":
        X1, n1, v1 = smiles_to_maccs(smiles)
        X2, n2, v2 = smiles_to_morgan(smiles)
        valid = v1 & v2
        X = np.hstack([X1, X2])
        names = n1 + n2
    elif feature_set == "morgan_rdkit":
        X1, n1, v1 = smiles_to_morgan(smiles)
        X2, n2, v2 = smiles_to_rdkit_descriptors(smiles)
        valid = v1 & v2
        X = np.hstack([X1, X2])
        names = n1 + n2
    elif feature_set == "maccs_morgan_rdkit":
        X1, n1, v1 = smiles_to_maccs(smiles)
        X2, n2, v2 = smiles_to_morgan(smiles)
        X3, n3, v3 = smiles_to_rdkit_descriptors(smiles)
        valid = v1 & v2 & v3
        X = np.hstack([X1, X2, X3])
        names = n1 + n2 + n3
    else:
        raise ValueError("feature_set bilinmiyor.")

    # Bu fonksiyonlarda tüm feature blokları aynı valid molekül sırasıyla üretildiği için
    # valid mask normalde aynı olur. Güvenlik için y ve df'yi maskeliyoruz.
    df_valid = df.loc[valid].reset_index(drop=True)
    y_valid = y[valid]

    note(
        "Feature matrisi üretildi",
        f"Feature set: {feature_set}\n"
        f"Geçerli molekül sayısı: {X.shape[0]}\n"
        f"Feature sayısı: {X.shape[1]}\n"
        f"İlk 5 feature: {names[:5]}"
    )

    return X, y_valid, names, df_valid


def make_rf(n_estimators=300, class_weight="balanced_subsample"):
    return RandomForestClassifier(
        n_estimators=n_estimators,
        max_features="sqrt",
        class_weight=class_weight,
        random_state=RANDOM_STATE,
        n_jobs=-1,
    )


def get_scores(model, X):
    if hasattr(model, "predict_proba"):
        return model.predict_proba(X)[:, 1]
    if hasattr(model, "decision_function"):
        return model.decision_function(X)
    return model.predict(X).astype(float)


def metric_dict(y_true, y_pred, y_score):
    cm = confusion_matrix(y_true, y_pred, labels=[0, 1])
    tn, fp, fn, tp = cm.ravel()
    specificity = tn / (tn + fp) if (tn + fp) else np.nan

    return {
        "ROC": roc_auc_score(y_true, y_score),
        "AP": average_precision_score(y_true, y_score),
        "F1": f1_score(y_true, y_pred, zero_division=0),
        "Accuracy": accuracy_score(y_true, y_pred),
        "BalancedAccuracy": balanced_accuracy_score(y_true, y_pred),
        "Recall": recall_score(y_true, y_pred, zero_division=0),
        "Specificity": specificity,
        "Precision": precision_score(y_true, y_pred, zero_division=0),
        "TN": int(tn), "FP": int(fp), "FN": int(fn), "TP": int(tp),
    }


def train_evaluate_model(model, X_train, X_test, y_train, y_test):
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    y_score = get_scores(model, X_test)
    return model, y_pred, y_score, metric_dict(y_test, y_pred, y_score)


def save_metrics_predictions(outdir, name, df_test, smiles_col, y_test, y_pred, y_score, metrics):
    outdir = Path(outdir)
    outdir.mkdir(parents=True, exist_ok=True)

    pred_df = pd.DataFrame({
        "SMILES": df_test[smiles_col].values if smiles_col in df_test.columns else np.arange(len(y_test)),
        "y_true": y_test,
        "y_pred": y_pred,
        "y_score_class1": y_score,
    })
    pred_df.to_csv(outdir / f"{name}_predictions.csv", index=False)
    pd.DataFrame([metrics]).to_csv(outdir / f"{name}_metrics.csv", index=False)

    print(f"[Kaydedildi] {outdir / f'{name}_predictions.csv'}")
    print(f"[Kaydedildi] {outdir / f'{name}_metrics.csv'}")


print("✅ Yardımcı fonksiyonlar hazır.")

## 5. SHAP, SHAP waterfall ve LIME ile model açıklama

Bu notebookta sadece skor bakmıyoruz; modelin neye göre karar verdiğini anlamaya çalışıyoruz.

Üreteceğimiz çıktılar:
- SHAP beeswarm
- SHAP bar
- SHAP waterfall
- LIME local explanation

In [ ]:
# SHAP ve LIME Colab'da kurulu değilse kurulacak.
import sys, subprocess, importlib.util

if importlib.util.find_spec("shap") is None:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "shap"])
if importlib.util.find_spec("lime") is None:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "lime"])

import shap
import lime
import lime.lime_tabular

lesson_out = OUTPUT_ROOT / "lesson08_interpretability_shap_lime"
lesson_out.mkdir(parents=True, exist_ok=True)

df, target_col, smiles_col, sep, source = load_dataset(ACTIVE_DATASET_KEY)
df_clean, y = prepare_target_and_smiles(df, target_col, smiles_col)

# SHAP için MACCS daha okunabilir feature isimleri verir.
X, y, feature_names, df_valid = build_feature_matrix(df_clean, y, smiles_col, feature_set="maccs")

X_train, X_test, y_train, y_test, df_train, df_test = train_test_split(
    X, y, df_valid, test_size=TEST_SIZE, stratify=y, random_state=RANDOM_STATE
)

rf = make_rf(n_estimators=500)
rf.fit(X_train, y_train)
y_pred = rf.predict(X_test)
y_score = get_scores(rf, X_test)

metrics = metric_dict(y_test, y_pred, y_score)
pd.DataFrame([metrics]).to_csv(lesson_out / "rf_metrics_for_interpretability.csv", index=False)
joblib.dump({"model": rf, "feature_names": feature_names}, lesson_out / "rf_model_for_shap_lime.joblib")

print("✅ RF modeli eğitildi.")
for k, v in metrics.items():
    print(f"{k}: {v:.4f}" if isinstance(v, float) else f"{k}: {v}")

In [ ]:
def extract_class1_shap(shap_values, expected_value):
    if isinstance(shap_values, list):
        values = shap_values[1] if len(shap_values) > 1 else shap_values[0]
    else:
        arr = np.asarray(shap_values)
        if arr.ndim == 3 and arr.shape[-1] == 2:
            values = arr[:, :, 1]
        elif arr.ndim == 3 and arr.shape[0] == 2:
            values = arr[1, :, :]
        else:
            values = arr

    ev = np.asarray(expected_value)
    if ev.ndim == 0:
        base_value = float(ev)
    elif ev.size > 1:
        base_value = float(ev.ravel()[1])
    else:
        base_value = float(ev.ravel()[0])
    return np.asarray(values, dtype=float), base_value

note("SHAP değerleri hesaplanıyor", "TreeExplainer Random Forest için uygundur.")

max_shap_samples = min(500, X_test.shape[0])
rng = np.random.RandomState(RANDOM_STATE)
shap_idx = rng.choice(X_test.shape[0], size=max_shap_samples, replace=False)
X_shap = X_test[shap_idx]

explainer = shap.TreeExplainer(rf)
raw_values = explainer.shap_values(X_shap)
shap_values, base_value = extract_class1_shap(raw_values, explainer.expected_value)

mean_abs = np.abs(shap_values).mean(axis=0)
shap_rank = pd.DataFrame({"Feature": feature_names, "mean_abs_SHAP": mean_abs}).sort_values("mean_abs_SHAP", ascending=False)
shap_rank.to_csv(lesson_out / "shap_top_features.csv", index=False)

print("✅ SHAP hesaplandı. Top 15 feature:")
display(shap_rank.head(15))

In [ ]:
plt.figure(figsize=(10, 8))
shap.summary_plot(shap_values, X_shap, feature_names=feature_names, max_display=25, show=False)
plt.title("SHAP Beeswarm — RF class 1")
plt.tight_layout()
plt.savefig(lesson_out / "shap_beeswarm_top25.png", dpi=300, bbox_inches="tight")
plt.show()

plt.figure(figsize=(10, 8))
shap.summary_plot(shap_values, X_shap, feature_names=feature_names, plot_type="bar", max_display=25, show=False)
plt.title("SHAP Bar — mean |SHAP|")
plt.tight_layout()
plt.savefig(lesson_out / "shap_bar_top25.png", dpi=300, bbox_inches="tight")
plt.show()

print("✅ SHAP beeswarm ve bar grafikleri kaydedildi.")

## 6. SHAP waterfall çizelim

Waterfall grafiği tek bir molekül için model tahmininin hangi feature’larla yukarı/aşağı itildiğini gösterir.

In [ ]:
def choose_waterfall_cases(y_true, y_pred, y_score):
    cases = []

    def add(label, mask, values, reverse=True):
        idx = np.where(mask)[0]
        if len(idx) == 0:
            return
        order = np.argsort(values[idx])
        if reverse:
            order = order[::-1]
        cases.append((label, int(idx[order[0]])))

    add("correct_positive_high_score", (y_true == 1) & (y_pred == 1), y_score, True)
    add("correct_negative_high_confidence", (y_true == 0) & (y_pred == 0), 1 - y_score, True)
    add("false_positive", (y_true == 0) & (y_pred == 1), y_score, True)
    add("false_negative", (y_true == 1) & (y_pred == 0), y_score, False)
    add("borderline_near_0p50", np.ones_like(y_true, dtype=bool), np.abs(y_score - 0.5), False)

    unique = []
    seen = set()
    for label, i in cases:
        if i not in seen:
            unique.append((label, i))
            seen.add(i)
    return unique

# SHAP değerlerini tüm test seti için hesaplayalım ki seçilen örneklerle indeksler uyumlu olsun.
raw_all = explainer.shap_values(X_test)
shap_all, base_all = extract_class1_shap(raw_all, explainer.expected_value)

exp = shap.Explanation(
    values=shap_all,
    base_values=np.repeat(base_all, X_test.shape[0]),
    data=X_test,
    feature_names=feature_names,
)

waterfall_rows = []
for label, i in choose_waterfall_cases(y_test, y_pred, y_score):
    plt.figure(figsize=(10, 7))
    shap.plots.waterfall(exp[i], max_display=15, show=False)
    plt.title(f"SHAP Waterfall — {label}\\ntrue={y_test[i]}, pred={y_pred[i]}, score={y_score[i]:.3f}")
    plt.tight_layout()
    out_file = lesson_out / f"shap_waterfall_{label}.png"
    plt.savefig(out_file, dpi=300, bbox_inches="tight")
    plt.show()

    waterfall_rows.append({
        "case": label,
        "test_index": i,
        "y_true": int(y_test[i]),
        "y_pred": int(y_pred[i]),
        "y_score": float(y_score[i]),
        "file": str(out_file),
    })

pd.DataFrame(waterfall_rows).to_csv(lesson_out / "shap_waterfall_cases.csv", index=False)
print("✅ SHAP waterfall grafikleri kaydedildi.")
display(pd.DataFrame(waterfall_rows))

## 7. LIME local explanation

LIME de tek örnek için açıklama üretir. SHAP’ten farklı bir yaklaşım kullanır.

In [ ]:
explainer_lime = lime.lime_tabular.LimeTabularExplainer(
    training_data=X_train,
    feature_names=feature_names,
    class_names=["class 0", "class 1"],
    mode="classification",
    discretize_continuous=False,
    random_state=RANDOM_STATE,
)

lime_rows = []
for case_no, (_, i) in enumerate(choose_waterfall_cases(y_test, y_pred, y_score)[:3], start=1):
    exp_lime = explainer_lime.explain_instance(
        data_row=X_test[i],
        predict_fn=rf.predict_proba,
        num_features=15,
        labels=(1,),
    )

    html_file = lesson_out / f"lime_case_{case_no}.html"
    exp_lime.save_to_file(str(html_file))

    weights = exp_lime.as_list(label=1)
    labels = [w[0] for w in weights]
    vals = [w[1] for w in weights]

    plt.figure(figsize=(10, 6))
    plt.barh(range(len(labels)), vals)
    plt.yticks(range(len(labels)), labels, fontsize=9)
    plt.axvline(0, color="black", linewidth=0.8)
    plt.gca().invert_yaxis()
    plt.xlabel("LIME weight for class 1")
    plt.title(f"LIME explanation — case {case_no}")
    plt.tight_layout()
    png_file = lesson_out / f"lime_case_{case_no}.png"
    plt.savefig(png_file, dpi=300, bbox_inches="tight")
    plt.show()

    for rank, (feature_rule, weight) in enumerate(weights, start=1):
        lime_rows.append({
            "case_no": case_no,
            "test_index": i,
            "rank": rank,
            "feature_rule": feature_rule,
            "lime_weight": weight,
            "html_file": str(html_file),
            "png_file": str(png_file),
        })

pd.DataFrame(lime_rows).to_csv(lesson_out / "lime_explanations.csv", index=False)
print("✅ LIME çıktıları kaydedildi.")